# 基礎断熱住宅の等級と隙間倍率の比較

外壁一般部の断熱材の熱抵抗に合わせて基礎外壁の外側にXPS 3種を配置します。1階床は合板・木部を保持し、床下の外気換気は停止します。等級は変更元の住宅モデルを示します。隙間倍率と断熱等級は独立に選べます。

基礎XPSの熱伝導率は0.028 W/(m·K)。計算上の厚さは等級4・5・6・7の順に47.3、102.4、122.4、157.4 mmです。外皮隙間倍率0、0.5、1、2、4、8のC値の目安は順に0、0.30、0.59、1.19、2.37、4.74 cm²/m²（元のColabでの対応）です。

In [ ]:
%pip -q install -U git+https://github.com/iguchi-lab/vtsimnx.git@be1e23daf5f51b4d1b3c3676ff2a6158acd8b515

ColabにHASP気象ファイルと元の「風圧係数.csv」を用意し、2つのパスを指定してください。APIのURLはColabシークレット `VTSIMNX_API_URL` から読みます。初期設定は自然室温です。全館空調では2階ホールから還気し、7系統に給気します。

In [ ]:
import pandas as pd
import vtsimnx as vt
from google.colab import userdata

h = vt.standard_house
weather_file = "/content/3639999.has"
coefficient_file = "/content/風圧係数.csv"
grades = [4, 5, 6, 7]
multipliers = [1]  # 6条件を比較するときは list(h.GAP_CASES)
central_air = False

In [ ]:
weather = vt.read_hasp(weather_file)
coefficients = pd.read_csv(coefficient_file, index_col=0)
pressure = h.facade_pressures(weather, coefficients)
solar = h.compute_solar(weather["直達日射量"], weather["水平面拡散日射量"])
nocturnal = h.compute_nocturnal(weather["夜間放射量"])
ground = vt.ground_temperature_by_depth(
    depth_m=1.0, t_out=weather["外気温"], solar_horizontal=solar["日射熱取得量（水平上向き）"],
    nocturnal_horizontal=weather["夜間放射量"], deep_layer_depth_m=10.0, deep_layer_temp_c=10.0,
    thermal_conductivity_w_mk=1.5, volumetric_heat_capacity_j_m3k=2.2e6,
    solar_to_surface_temp_coeff=0.003, nocturnal_to_surface_temp_coeff=0.003, spinup=True,
)

In [ ]:
def make_case(grade, multiplier):
    key = f"foundation{grade}"
    surfaces = h.bind_surface_types(h.surface_types[key], h.layers[key], solar, nocturnal)
    nodes = h.gap_nodes(weather["外気温"], weather["外気絶対湿度"] / 1000, pressure, central_air=central_air)
    return {
        "simulation": {"index": weather.index, "warmup": {"start": h.WARMUP_START}},
        "nodes": [*nodes, {"key": "地下1m", "t": ground}],
        "surfaces": h.compose_surfaces(h.surface_rows[key], surfaces),
        "ventilation_branches": h.gap_network(multiplier, central_air=central_air),
        "aircon": h.central_aircon() if central_air else [],
        "heat_source": h.heat_source["default"], "humidity_source": h.humidity_source["default"],
    }

In [ ]:
url = userdata.get("VTSIMNX_API_URL")
summary = []
for grade in grades:
    for multiplier in multipliers:
        result = vt.run_calc(url, make_case(grade, multiplier))
        temp = result.get_series_df("thermal_temperature")
        humidity = result.get_series_df("humidity_x") * 1000
        flow = result.get_series_df("vent_flow_rate") * 3600
        prefix = f"foundation{grade}_gap{multiplier}"
        temp.to_csv(prefix + "_室温.csv")
        humidity.to_csv(prefix + "_絶対湿度.csv")
        flow.to_csv(prefix + "_風量.csv")
        summary.append({"等級": grade, "外皮隙間倍率": multiplier,
                        "LD平均室温 [℃]": temp["LD"].mean(), "LD平均絶対湿度 [g/kg(DA)]": humidity["LD"].mean()})
pd.DataFrame(summary)